<a href="https://colab.research.google.com/github/deeedaniel/gpt/blob/main/buildgpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import urllib.request

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

torch.manual_seed(1337)

cuda


In [3]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

urllib.request.urlretrieve(url, "input.txt")

with open("input.txt", "r") as f:
    text = f.read()

print(len(text))

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(vocab_size, chars)

1115394
65 ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [4]:
!pip install tokenizers -q
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"], vocab_size=1000)
tokenizer.train(files=["input.txt"], trainer=trainer)
tokenizer.save("shakespeare-bpe.json")

vocab_size = tokenizer.get_vocab_size()

def encode(s):
    return tokenizer.encode(s).ids

def decode(l):
    return tokenizer.decode(l)


[41, 39, 58]
cat


In [5]:
# Convert our text into data (tensor)
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:20])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56])


In [6]:
# Split data into training data and validating data
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
print(len(train_data), len(val_data))

1003854 111540


In [7]:
block_size = 32
batch_size = 16

def get_batch(split):
  data = train_data if split == 'train' else val_data

  # pick x (4) random starting indices for our batches
  ix = torch.randint(len(data) - block_size, (batch_size,))

  # for all starting portions, get 8 consecutive characters
  # stack them up into 4 separate chunks for different batches
  # (4,8) 4 batch items, 8 chars each
  x = torch.stack([data[i:i+block_size] for i in ix])

  # y is the exact same but shifted forward by 1 to represent correct next char
  y = torch.stack([data[i+1:i+1+block_size] for i in ix])

  x, y = x.to(device), y.to(device)
  return x,y

In [8]:
xb, yb = get_batch('train')
print(xb.shape, yb.shape)
print(xb[0])
print(yb[0])
print(decode(xb[0].tolist()))
print(decode(yb[0].tolist()))

torch.Size([16, 32]) torch.Size([16, 32])
tensor([58, 53,  1, 41, 53, 56, 56, 59, 54, 58,  1, 39,  1, 51, 39, 52,  5, 57,
         1, 61, 47, 44, 43,  1, 47, 57,  0, 61, 46, 43, 52,  1],
       device='cuda:0')
tensor([53,  1, 41, 53, 56, 56, 59, 54, 58,  1, 39,  1, 51, 39, 52,  5, 57,  1,
        61, 47, 44, 43,  1, 47, 57,  0, 61, 46, 43, 52,  1, 57],
       device='cuda:0')
to corrupt a man's wife is
when 
o corrupt a man's wife is
when s


In [9]:
class BigramModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

  def forward(self, idx, targets=None):
    logits = self.token_embedding_table(idx)

    loss = None
    if targets is not None:
      B,T,C = logits.shape

      # Flatten batch and time to compare logits and targets
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

model = BigramModel(vocab_size).to(device)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

torch.Size([512, 65])
tensor(4.6057, device='cuda:0', grad_fn=<NllLossBackward0>)


In [10]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

batch_size = 4

for step in range(5000):
  xb,yb = get_batch('train')
  logits, loss = model(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

  if step % 500 == 0:
    print(f"step {step}: loss {loss.item():.4f}")

print("final loss: ", loss.item())

step 0: loss 4.4447
step 500: loss 4.1006
step 1000: loss 3.7359
step 1500: loss 3.3723
step 2000: loss 3.0913
step 2500: loss 3.1310
step 3000: loss 2.8182
step 3500: loss 3.0432
step 4000: loss 2.6842
step 4500: loss 2.7078
final loss:  2.7302451133728027


In [11]:
def generate(model, idx, max_new_tokens):
  for _ in range(max_new_tokens):
    idx_con = idx[:, -block_size:]
    logits, loss = model(idx_con)
    logits = logits[:, -1, :]
    probs = F.softmax(logits, dim=-1)
    idx_next = torch.multinomial(probs, num_samples=1)
    idx = torch.cat((idx, idx_next), dim=1)
  return idx

In [12]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)  # start with a single "character 0" as a seed
generated = generate(model, context, max_new_tokens=50)
print(decode(generated[0].tolist()))




CExbikRZkid owiaZm, s s, btK

Hiset bBayou e.
Sv


In [13]:
embed_dim = 64
head_size = 8

class Head(nn.Module):
  def __init__(self, embed_dim, head_size, block_size):
    super().__init__()
    self.key = nn.Linear(embed_dim, head_size, bias=False, device=device)
    self.query = nn.Linear(embed_dim, head_size, bias=False, device=device)
    self.value = nn.Linear(embed_dim, head_size, bias=False, device=device)
    self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

  def forward(self, x):
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)
    v = self.value(x)

    weights = q @ k.transpose(-2,-1) * (head_size ** -0.5)
    weights = weights.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
    weights = F.softmax(weights, dim=-1)

    out = weights @ v
    return out


In [14]:
token_embedding_table = nn.Embedding(vocab_size, embed_dim, device=device)
x = token_embedding_table(xb)   # xb from before, shape (32... wait, actually (4,8) if you kept batch_size=4 for xb, or reflect your current batch_size)
print(x.shape)

head = Head(embed_dim, head_size, block_size).to(device)
out = head(x)
print(out.shape)

torch.Size([4, 32, 64])
torch.Size([4, 32, 8])


In [15]:
num_heads = 4

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, embed_dim, head_size, block_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(embed_dim, head_size, block_size) for _ in range(num_heads)])
    self.proj = nn.Linear(num_heads * head_size, embed_dim)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.proj(out)
    return out

In [16]:
mha = MultiHeadAttention(num_heads, embed_dim, head_size, block_size).to(device)
out = mha(x)
print(out.shape)

torch.Size([4, 32, 64])


In [17]:
class FeedForward(nn.Module):
  def __init__(self, embed_dim):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(embed_dim, 4 * embed_dim),
        nn.ReLU(),
        nn.Linear(4*embed_dim, embed_dim),
    )

  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, embed_dim, num_heads, block_size):
    super().__init__()
    head_size = embed_dim // num_heads
    self.sa = MultiHeadAttention(num_heads, embed_dim, head_size, block_size)
    self.ffwd = FeedForward(embed_dim)
    self.ln1 = nn.LayerNorm(embed_dim)
    self.ln2 = nn.LayerNorm(embed_dim)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x

In [18]:
block = Block(embed_dim, num_heads, block_size).to(device)
out = block(x)
print(out.shape)

torch.Size([4, 32, 64])


In [19]:
num_layers = 4

class TinyGPT(nn.Module):
  def __init__(self, vocab_size, embed_dim, block_size, num_heads, num_layers):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, embed_dim, device=device)
    self.position_embedding_table = nn.Embedding(block_size, embed_dim, device=device)
    self.blocks = nn.Sequential(*[Block(embed_dim, num_heads, block_size) for _ in range(num_layers)]).to(device)
    self.ln_f = nn.LayerNorm(embed_dim).to(device)
    self.lm_head = nn.Linear(embed_dim, vocab_size).to(device)

  def forward(self, idx, targets=None):
    B,T = idx.shape
    tok_emb = self.token_embedding_table(idx)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device))
    x = tok_emb + pos_emb
    x = self.blocks(x)
    x = self.ln_f(x)
    logits = self.lm_head(x)

    loss = None
    if targets is not None:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

In [34]:
model = TinyGPT(vocab_size, embed_dim, block_size, num_heads, num_layers).to(device)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

torch.Size([128, 1000])
tensor(7.0621, device='cuda:0', grad_fn=<NllLossBackward0>)


In [36]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(10000):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 500 == 0:
        print(f"step {step}: loss {loss.item():.4f}")

print("final loss:", loss.item())

step 0: loss 7.0931
step 500: loss 5.3929
step 1000: loss 4.4052
step 1500: loss 4.6069
step 2000: loss 4.2676
step 2500: loss 4.6093
step 3000: loss 3.8118
step 3500: loss 4.1875
step 4000: loss 4.1755
step 4500: loss 3.3430
step 5000: loss 3.3341
step 5500: loss 3.9826
step 6000: loss 3.7799
step 6500: loss 4.2522
step 7000: loss 3.5008
step 7500: loss 4.0003
step 8000: loss 3.7820
step 8500: loss 3.6613
step 9000: loss 3.6513
step 9500: loss 4.2796
final loss: 3.638700485229492


In [37]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = generate(model, context, max_new_tokens=1000)
print(decode(generated[0].tolist()))

 as present must so,
Then call our we penit forfeit hands
Plooking theix initing soul town; a madequarWe
our lother: what call me, have nurse druly rood Xccurlish
th'd not he is our uncapected; take of our oaths
nerleumble that perpostion of Clauptivily,
To make hope you obed other me? of shost me;
Heivent like himself of my teny, and my shipt.

FLORGE LAURGITA:
I can speak with terrous hardward and wretcheder
Not where I that a tortest would to i'eni' my gracious
As to cree selst thou KING London sight, not emerate
I love thy love with peace in me; 'tis ainter
Is reading upon the tail; but he my death'sly news
I war to Wive high passionly bitters, welieve my
ail Caius after of my house, noon!

QUEEN ELIZABETH:
Whyly, thereforeast of Gey,
Cewels be some other traistress city or the chances I did
Grow a blood so I could trustvour with flame up on the body
Anon prat: yet if I mought myself there my death
Who pleror until you seem to be you. I cannot know you
Now, babelst of heart. 
All! 